# Homework 4: Logistic Regression and Feature Scaling

This week, I will be using logistic regression to predict the probabilities of the target. I have a feeling this model will perform better than the linear regression models since the targets are binary, which logistic regression is designed for. When it comes to feature scaling, I have already scaled some features within my data due to the large outliers they contained when I began working with them last semester.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

## Dataset 1: Instacart Analysis

In [2]:
data1 = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/instacart.csv')

transform_cols = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase']

for col in transform_cols:
    data1[f'log_{col}'] = np.log1p(data1[col])

data1_transformed = data1.drop(columns = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase'])

data1_transformed.head()

,user_id,product_id,ordered,log_times_purchased,log_num_orders,log_frequency,log_Product Popularity,log_Last Product Order,log_Last Order,log_Orders Since Last Purchase
0,71,45,0.0,1.791759,3.178054,0.196710,0.005878,2.302585,3.178054,2.708050
1,71,117,1.0,2.995732,3.178054,0.602175,0.001219,3.178054,3.178054,0.000000
2,71,2078,0.0,0.693147,3.178054,0.042560,0.006478,1.386294,3.178054,3.044522
3,71,2825,0.0,1.098612,3.178054,0.083382,0.003743,1.945910,3.178054,2.890372
4,71,3376,1.0,1.098612,3.178054,0.083382,0.003602,2.995732,3.178054,1.609438


In [10]:
X = data1_transformed.drop(columns = ['ordered', 'user_id', 'product_id'])
y = data1_transformed['ordered']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

model = LogisticRegression()

model.fit(X_train, y_train)

y_preds = model.predict_proba(X_test)[:, 1]

values = pd.DataFrame({'Actual' : y_test, 'Predicted' : y_preds})

values['Decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('Decile').agg(avg_probability = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = values['Actual'].mean()

lift_table['Lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_probability'] = lift_table['avg_probability'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['Lift'] = lift_table['Lift'].map(lambda x: f'{x:.3f}')

display(lift_table)

,avg_probability,avg_ordered,Lift
Decile,,,
1,0.005,0.006,0.088
2,0.010,0.008,0.125
3,0.015,0.013,0.205
4,0.021,0.023,0.355
5,0.028,0.030,0.474
6,0.038,0.044,0.691
7,0.053,0.058,0.909
8,0.076,0.076,1.194
9,0.113,0.117,1.847


Comparing these results to the regressions performed in Week 2, I found that Logistic Regression did a tiny bit better than the ridge regression model. The small increase in performance does surprise me since I assumed it would have done much better. I also am noticing that the average for ordered and probability are both extremely low. This is most likely due to the large imbalance of customers that do not order products.

## Dataset 2: Amazon Reviews Dataset

In [11]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

small_reviews = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_reviews%5B1%5D.csv')

small_meta = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_meta%5B1%5D.csv')

small_meta = small_meta.set_index(keys = 'parent_asin')

small_meta = small_meta.drop(columns = ['title', 'images'])

data2 = small_reviews.join(small_meta, on = 'parent_asin', how = 'inner')

data2 = data2.drop(columns = ['main_category', 'store', 'rating_number', 'videos', 'images', 'subtitle', 'author', 'bought_together', 'features', 'description', 'details', 'timestamp', 'text', 'title'])

data2['product_review_count'] = data2.groupby('asin').transform('size')
data2['user_review_count'] = data2.groupby('user_id').transform('size')

data2['price'] = data2['price'].fillna(data2['price'].median())

data2['categories'] = data2['categories'].apply(ast.literal_eval)

mlb = MultiLabelBinarizer()

categories = pd.DataFrame(mlb.fit_transform(data2['categories']), columns = mlb.classes_, index = data2.index)

top_30 = categories.sum().sort_values(ascending = False).head(31).index

categories = categories[top_30[1:31]]

data2 = data2.drop(columns = 'categories')

data2 = data2.join(categories)

columns_to_transform = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count']

for col in columns_to_transform:
    data2[f'log_{col}'] = np.log1p(data2[col])

data2_transformed = data2.drop(columns = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count'])

data2_transformed = data2_transformed.reset_index(drop = True)

for i in range(len(data2_transformed)):
    if data2_transformed.iloc[i]['rating'] == 4:
        data2_transformed.loc[i, 'liked'] = 1
    elif data2_transformed.iloc[i]['rating'] == 5:
        data2_transformed.loc[i, 'liked'] = 1
    else:
        data2_transformed.loc[i, 'liked'] = 0

X = data2_transformed.drop(columns = ['asin', 'parent_asin', 'user_id', 'liked'])
y = data2_transformed['liked']

X.head()

,rating,verified_purchase,Beverages,Snacks & Sweets,Pantry Staples,Coffee,Snack Foods,Candy & Chocolate,"Bottled Beverages, Water & Drink Mixes",Cooking & Baking,...,Whole Coffee Beans,"Canned, Jarred & Packaged Foods",Cold Cereals,"Baking Syrups, Sugars & Sweeteners",Black,log_helpful_vote,log_average_rating,log_price,log_product_review_count,log_user_review_count
0,5,False,1,0,0,0,0,0,0,0,...,0,0,0,0,1,0.000000,1.704748,3.257712,0.693147,0.693147
1,1,True,1,0,0,0,0,0,0,0,...,0,0,0,0,0,3.295837,1.686399,3.351307,0.693147,0.693147
2,4,True,1,0,0,1,0,0,0,0,...,1,0,0,0,0,0.000000,1.686399,2.889816,0.693147,1.098612
3,5,True,0,0,1,0,0,0,0,1,...,0,0,0,0,0,0.000000,1.722767,2.771964,0.693147,1.098612
4,5,True,1,0,0,1,0,0,0,0,...,0,0,0,0,0,0.000000,1.686399,3.395179,1.609438,0.693147


In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

model = LogisticRegression()

model.fit(X_train, y_train)

y_preds = model.predict_proba(X_test)[:, 1]

values = pd.DataFrame({'Actual' : y_test, 'Predicted' : y_preds})

values['Decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('Decile').agg(avg_probability = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = values['Actual'].mean()

lift_table['Lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_probability'] = lift_table['avg_probability'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['Lift'] = lift_table['Lift'].map(lambda x: f'{x:.3f}')

display(lift_table)

,avg_probability,avg_ordered,Lift
Decile,,,
1,0.011,0.000,0.000
2,0.472,0.400,0.479
3,0.975,1.000,1.197
4,0.998,1.000,1.197
5,0.999,1.000,1.197
6,0.999,1.000,1.197
7,0.999,1.000,1.197
8,0.999,1.000,1.197
9,0.999,1.000,1.197


The logistic regression has the same performance as it did for the ridge regression from Week 2. This isn't very surprising due to the fact that the majority of the instances are liked due to the large class imbalance, and the populaiton is small. The population for this dataset has to be small due to the maximum file size github allows for uploaded files. I had to select a much smaller subset of datapoints to ensure I could use the dataset.

## Dataset 3: Ecommerce Dataset

In [13]:
data3 = pd.read_csv('/workspaces/DX799---HW-Assignments/data.csv', encoding = 'windows-1252')
data3['Purchased'] = 1

customer_id = data3['Customer ID'].values
segments = data3['Segment'].values
cities = data3['City'].values
state = data3['State'].values
region = data3['Region'].values
product_id = data3['Product ID'].values
categories = data3['Category'].values
subcategories = data3['Sub-Category'].values

def generate_row(df):
    row = {}

    row['Customer ID'] = np.random.choice(customer_id)
    cust_id = row['Customer ID']
    purchased = set(df[df['Customer ID'] == cust_id]['Product ID'])
    candidates = list(set(product_id) - purchased)

    row['Segment'] = np.random.choice(segments)
    row['State'] = np.random.choice(state)
    row['City'] = np.random.choice(cities)
    row['Region'] = np.random.choice(region)
    row['Product ID'] = np.random.choice(candidates)
    row['Category'] = np.random.choice(categories)
    row['Sub-Category'] = np.random.choice(subcategories)

    row['Sales'] = 0
    row['Discount'] = 0
    row['Quantity'] = 0
    row['Profit'] = 0
    row['Purchased'] = 0

    return row

fake_rows = pd.DataFrame([generate_row(data3) for n in range(1000)])

fake_df = pd.concat([data3, fake_rows], ignore_index = True)

fake_df = fake_df.drop(columns = ['Row ID', 'Product Name', 'Country', 'Postal Code', 'Order ID', 'Order Date'])

fake_df_encoded = pd.get_dummies(fake_df, columns = ['Ship Mode', 'Segment', 'State', 'Region', 'Category', 'Sub-Category'], dtype = int)

X = fake_df_encoded.drop(columns = ['Customer ID', 'Product ID', 'City', 'Purchased'])
y = fake_df_encoded['Purchased']

X.head()

,Sales,Quantity,Discount,Profit,Ship Mode_First Class,Ship Mode_Same Day,Ship Mode_Second Class,Ship Mode_Standard Class,Segment_Consumer,Segment_Corporate,...,Sub-Category_Envelopes,Sub-Category_Fasteners,Sub-Category_Furnishings,Sub-Category_Labels,Sub-Category_Machines,Sub-Category_Paper,Sub-Category_Phones,Sub-Category_Storage,Sub-Category_Supplies,Sub-Category_Tables
0,48.896,4,0.2,8.5568,0,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,0
1,474.430,11,0.0,199.2606,0,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,0
2,3.600,2,0.0,1.7280,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,454.560,5,0.2,-107.9580,0,0,0,1,1,0,...,0,0,0,0,0,0,0,1,0,0
4,141.420,5,0.6,-187.3815,0,0,0,1,1,0,...,0,0,1,0,0,0,0,0,0,0


In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

model = LogisticRegression()

model.fit(X_train, y_train)

y_preds = model.predict_proba(X_test)[:, 1]

values = pd.DataFrame({'Actual' : y_test, 'Predicted' : y_preds})

values['Decile'] = pd.qcut(values['Predicted'].rank(method = 'first'), q = 10, labels = False) + 1

lift_table = values.groupby('Decile').agg(avg_probability = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = values['Actual'].mean()

lift_table['Lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_probability'] = lift_table['avg_probability'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['Lift'] = lift_table['Lift'].map(lambda x: f'{x:.3f}')

display(lift_table)

,avg_probability,avg_ordered,Lift
Decile,,,
1,0.002,0.000,0.000
2,0.003,0.000,0.000
3,0.839,0.843,1.075
4,1.000,1.000,1.276
5,1.000,1.000,1.276
6,1.000,1.000,1.276
7,1.000,1.000,1.276
8,1.000,1.000,1.276
9,1.000,1.000,1.276


The results for logistic regression once again are the same as the ridge regression for this model. I again believe this is due to a combination of small population and large class imbalance.

## Conclusion

For all of these models, logistic regression performed just as well or a little better than the ridge regression model which was the prevailing model in Week 2. For all my datasets, I do know there is a large class imbalance, so I will balance the data when I begin choosing my final model for this project. I also want to look into finding a way to create better fake instances for dataset three since I feel that is also contributing to the high performance.